In [ ]:
import os
import time
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
import matplotlib.pyplot as plt
from tqdm import tqdm

import timm  # pip install timm
from torch.amp import autocast, GradScaler


# ==========================================
# [증강] CutMix 유틸
# ==========================================
def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1, bby1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
    bbx2, bby2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2


# ==========================================
# [EMA] 버퍼(BN 통계) 복사 + Dynamic Decay
# ==========================================
class ModelEMA:
    def __init__(self, model, decay=0.9998):
        self.ema = copy.deepcopy(model).eval()
        self.decay = decay
        self.updates = 0
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        with torch.no_grad():
            for ema_p, p in zip(self.ema.parameters(), model.parameters()):
                ema_p.data.mul_(d).add_(p.data, alpha=1 - d)
            for ema_b, b in zip(self.ema.buffers(), model.buffers()):
                ema_b.data.copy_(b.data)


def get_parameter_groups(model, weight_decay=0.05):
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if param.ndim == 1 or name.endswith(".bias"):
            no_decay.append(param)
        else:
            decay.append(param)
    return [{'params': no_decay, 'weight_decay': 0.0},
            {'params': decay, 'weight_decay': weight_decay}]


# ==========================================
# [평가] clean train / val 공용 (TTA 옵션)
# ==========================================
@torch.no_grad()
def evaluate(model, loader, criterion, device, tta=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        with autocast('cuda'):
            outputs = model(inputs)
            if tta:
                outputs = (outputs + model(torch.flip(inputs, dims=[3]))) / 2.0
            loss = criterion(outputs, labels)
        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return total_loss / total, correct / total


# ==========================================
# [훈련]
# ==========================================
def train_model(model, train_loader, clean_train_loader, val_loader,
                criterion, optimizer, scheduler, scaler, device, epochs, save_path):
    best_acc = 0.0
    ema = ModelEMA(model, decay=0.9998)
    history = {'clean_train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

    print("\n[파인튜닝] ConvNeXt-Tiny (ImageNet-22k pretrained) + EMA/CutMix/warmup-cosine/TTA")
    print("사전학습 가중치 보호를 위해 LR을 낮춰 시작합니다.\n")

    for epoch in range(epochs):
        start_time = time.time()

        # ---------------- 훈련 ----------------
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1:03d}/{epochs:03d} [Train]")

        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)

            apply_cutmix = np.random.rand() < 0.5
            if apply_cutmix:
                lam = np.random.beta(1.0, 1.0)
                rand_index = torch.randperm(inputs.size(0), device=device)
                target_a, target_b = labels, labels[rand_index]
                bbx1, bby1, bbx2, bby2 = rand_bbox(inputs.size(), lam)
                inputs[:, :, bbx1:bbx2, bby1:bby2] = inputs[rand_index, :, bbx1:bbx2, bby1:bby2]
                lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (inputs.size(-1) * inputs.size(-2)))

            with autocast('cuda'):
                outputs = model(inputs)
                if apply_cutmix:
                    loss = criterion(outputs, target_a) * lam + criterion(outputs, target_b) * (1. - lam)
                else:
                    loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            ema.update(model)

            running_loss += loss.item() * inputs.size(0)
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        epoch_train_loss = running_loss / len(train_loader.dataset)

        # ---- 진단: clean train acc (증강 X, EMA 모델) ----
        _, clean_train_acc = evaluate(ema.ema, clean_train_loader, criterion, device, tta=False)
        # ---- 검증 (EMA + TTA) ----
        epoch_val_loss, epoch_val_acc = evaluate(ema.ema, val_loader, criterion, device, tta=True)

        scheduler.step()

        history['clean_train_acc'].append(clean_train_acc)
        history['val_acc'].append(epoch_val_acc)
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)

        gap = clean_train_acc - epoch_val_acc
        elapsed = time.time() - start_time
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch + 1:03d} | {elapsed:.0f}s | LR {lr:.6f} | "
              f"CleanTrain {clean_train_acc:.4f} | Val {epoch_val_acc:.4f} | Gap {gap:+.4f}")

        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            torch.save(ema.ema.state_dict(), save_path)
            print(f"  >> best 갱신 · 저장: {save_path}\n")
        else:
            print()

    print(f"\n훈련 종료. 최고 검증 정확도(EMA): {best_acc:.4f}")
    return history


def plot_history(history, save_dir):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history['clean_train_acc'], label='Clean Train Acc')
    plt.plot(history['val_acc'], label='Val Acc (EMA+TTA)')
    plt.title('Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Loss')
    plt.legend()
    graph_path = os.path.join(save_dir, "finetune_log.png")
    plt.savefig(graph_path)
    print(f"\n그래프 저장: {graph_path}")
    try:
        plt.show()
    except Exception:
        pass


# ==========================================
# [메인]
# ==========================================
if __name__ == "__main__":
    PATH_LOCAL = r"C:\Users\user\Desktop\졸작_최종_파이프라인"
    PATH_ONEDRIVE = r"C:\Users\user\OneDrive\바탕 화면\졸작_최종_파이프라인"
    DATA_DIR = PATH_LOCAL if os.path.exists(PATH_LOCAL) else PATH_ONEDRIVE
    MODEL_SAVE_PATH = os.path.join(DATA_DIR, "best_convnext_model.pt")

    BATCH_SIZE = 64
    EPOCHS = 40           # 사전학습 파인튜닝은 수렴이 빨라 100까지 필요 없음
    LEARNING_RATE = 1e-4  # from-scratch 5e-4 → 파인튜닝은 낮춰야 사전학습 지식이 안 망가짐
    WARMUP_EPOCHS = 3
    NUM_WORKERS = 4       # 프리징 나면 0으로

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"학습 장치: {device}")

    # 파인튜닝은 증강을 살짝 약하게 (사전학습 표현을 크게 흔들지 않도록)
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.1),
    ])
    val_transform = transforms.Compose([
        transforms.Resize(236),           # crop_pct 0.95 근사 (224/0.95≈236)
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_root = os.path.join(DATA_DIR, 'train')
    val_root = os.path.join(DATA_DIR, 'val')

    print("데이터셋 로딩 중...")
    train_dataset = datasets.ImageFolder(root=train_root, transform=train_transform)
    val_dataset = datasets.ImageFolder(root=val_root, transform=val_transform)

    clean_full = datasets.ImageFolder(root=train_root, transform=val_transform)
    random.seed(42)
    clean_idx = random.sample(range(len(clean_full)), min(3000, len(clean_full)))
    clean_train_dataset = Subset(clean_full, clean_idx)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0), pin_memory=True)
    clean_train_loader = DataLoader(clean_train_dataset, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0), pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0), pin_memory=True)

    NUM_CLASSES = len(train_dataset.classes)
    print(f"클래스 수: {NUM_CLASSES}")

    # ---- 백본 교체: ImageNet-22k 사전학습 ConvNeXt-Tiny ----
    print("ConvNeXt-Tiny(fb_in22k_ft_in1k) 사전학습 가중치 로딩...")
    model = timm.create_model(
        'convnext_tiny.fb_in22k_ft_in1k',
        pretrained=True,
        num_classes=NUM_CLASSES,
        drop_path_rate=0.1,       # DropPath는 timm 인자로 처리
    ).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    param_groups = get_parameter_groups(model, weight_decay=0.05)
    optimizer = optim.AdamW(param_groups, lr=LEARNING_RATE)

    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS)
    cosine = CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])

    scaler = GradScaler('cuda')

    history = train_model(model, train_loader, clean_train_loader, val_loader,
                          criterion, optimizer, scheduler, scaler, device, EPOCHS, MODEL_SAVE_PATH)
    plot_history(history, DATA_DIR)


In [ ]:
"""
ConvNeXt-Tiny 파인튜닝 v2  (규제 강화 + 데이터 누수 검사 통합)

MODE 설정으로 실행 모드를 고릅니다.
  "leakcheck" : train/val 사이 near-duplicate(데이터 누수)만 검사하고 종료
  "train"     : 학습만 실행
  "both"      : 누수 검사 후 이어서 학습

v1(88.23%) 대비 변경점
  - 증강 강화 : RandomResizedCrop scale 0.8→0.65, ColorJitter→TrivialAugmentWide,
                RandomErasing 0.1→0.25
  - Mixup 추가 (기존 CutMix와 확률적으로 번갈아 적용)
  - drop_path_rate 0.1 → 0.3
  - Layer-wise LR Decay(0.75) 적용 : 하위 레이어는 낮은 LR로 사전학습 지식 보호
  - epochs 40 → 50 (규제가 세지면 수렴이 느려짐)
"""

import os
import time
import copy
import random
import csv
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from torch.amp import autocast, GradScaler
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import timm


# ==========================================================
# 설정
# ==========================================================
MODE = "both"          # "leakcheck" | "train" | "both"

PATH_LOCAL = r"C:\Users\user\Desktop\졸작_최종_파이프라인"
PATH_ONEDRIVE = r"C:\Users\user\OneDrive\바탕 화면\졸작_최종_파이프라인"
DATA_DIR = PATH_LOCAL if os.path.exists(PATH_LOCAL) else PATH_ONEDRIVE

MODEL_SAVE_PATH = os.path.join(DATA_DIR, "best_convnext_v2.pt")
LEAK_REPORT_PATH = os.path.join(DATA_DIR, "leak_report.csv")

BATCH_SIZE = 64
EPOCHS = 50
BASE_LR = 1.5e-4        # head 기준. 하위 레이어는 LLRD로 자동 감쇠
LAYER_DECAY = 0.75
WEIGHT_DECAY = 0.05
DROP_PATH = 0.3
WARMUP_EPOCHS = 3
NUM_WORKERS = 4         # 프리징 나면 0

# 누수 검사 파라미터
HAMMING_THRESH = 5      # pHash 해밍거리 <= 5 면 '사실상 동일 이미지'로 간주
HASH_WORKERS = 8


# ==========================================================
# [A] 데이터 누수 검사 (pHash 기반 near-duplicate 탐지)
# ==========================================================
_POPCOUNT = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)


def _dct_matrix(n):
    idx = np.arange(n)
    k = idx.reshape(-1, 1)
    m = np.cos(np.pi * (2 * idx + 1) * k / (2 * n))
    m[0] *= 1 / np.sqrt(2)
    return m * np.sqrt(2 / n)


_DCT32 = _dct_matrix(32)


def phash(path):
    """32x32 그레이스케일 DCT 기반 64bit perceptual hash."""
    try:
        img = Image.open(path)
        img.draft('L', (32, 32))          # JPEG 디코딩 가속 (핵심 최적화)
        img = img.convert('L').resize((32, 32), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float64)
        d = _DCT32 @ arr @ _DCT32.T
        low = d[:8, :8].flatten()
        med = np.median(low[1:])          # DC 성분 제외한 중앙값
        bits = (low > med).astype(np.uint8)
        return np.packbits(bits).view(np.uint64)[0]
    except Exception:
        return None


def collect_files(root):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    files = []
    for dirpath, _, names in os.walk(root):
        for n in names:
            if os.path.splitext(n)[1].lower() in exts:
                files.append(os.path.join(dirpath, n))
    return sorted(files)


def hash_all(files, desc):
    hashes = [None] * len(files)
    with ThreadPoolExecutor(max_workers=HASH_WORKERS) as ex:
        for i, h in enumerate(tqdm(ex.map(phash, files), total=len(files), desc=desc)):
            hashes[i] = h
    keep = [(f, h) for f, h in zip(files, hashes) if h is not None]
    return [f for f, _ in keep], np.array([h for _, h in keep], dtype=np.uint64)


def run_leak_check(data_dir, report_path):
    train_root = os.path.join(data_dir, 'train')
    val_root = os.path.join(data_dir, 'val')

    print("\n" + "=" * 60)
    print("[A] 데이터 누수 검사 : train/val 간 near-duplicate 탐지")
    print("=" * 60)

    train_files = collect_files(train_root)
    val_files = collect_files(val_root)
    print(f"train {len(train_files):,}장 / val {len(val_files):,}장")

    train_files, train_h = hash_all(train_files, "hash(train)")
    val_files, val_h = hash_all(val_files, "hash(val)")

    matches = []
    chunk = 64
    for s in tqdm(range(0, len(val_h), chunk), desc="compare"):
        vb = val_h[s:s + chunk]
        xor = vb[:, None] ^ train_h[None, :]
        pc = _POPCOUNT[xor.view(np.uint8).reshape(xor.shape[0], xor.shape[1], 8)].sum(axis=2)
        hits = np.argwhere(pc <= HAMMING_THRESH)
        for r, c in hits:
            matches.append((val_files[s + int(r)], train_files[int(c)], int(pc[r, c])))

    leaked_val = {m[0] for m in matches}
    ratio = len(leaked_val) / max(len(val_files), 1) * 100

    with open(report_path, 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.writer(f)
        w.writerow(['val_image', 'train_duplicate', 'hamming'])
        w.writerows(matches)

    print(f"\n중복 쌍 {len(matches):,}건 / 오염된 val 이미지 {len(leaked_val):,}장 "
          f"= val의 {ratio:.2f}%")
    print(f"상세 리포트: {report_path}")
    if ratio < 1:
        print("판정: 누수 거의 없음. 현재 val 점수는 신뢰 가능합니다.")
    elif ratio < 5:
        print("판정: 경미한 누수. 점수가 소폭(1%p 내외) 부풀려졌을 수 있습니다.")
    else:
        print("판정: 심각한 누수. 상품 단위 재분할이 필요하며 현재 점수는 과대평가입니다.")
    print("=" * 60 + "\n")
    return ratio


# ==========================================================
# [B] Mixup / CutMix
# ==========================================================
def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1, bby1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
    bbx2, bby2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2


def mix_batch(inputs, labels, mixup_alpha=0.8, cutmix_alpha=1.0,
              prob=0.8, switch_prob=0.5):
    """확률 prob로 Mixup 또는 CutMix 적용. (inputs, target_a, target_b, lam) 반환"""
    if np.random.rand() > prob:
        return inputs, labels, labels, 1.0

    idx = torch.randperm(inputs.size(0), device=inputs.device)
    target_a, target_b = labels, labels[idx]

    if np.random.rand() < switch_prob:      # CutMix
        lam = np.random.beta(cutmix_alpha, cutmix_alpha)
        x1, y1, x2, y2 = rand_bbox(inputs.size(), lam)
        inputs[:, :, x1:x2, y1:y2] = inputs[idx, :, x1:x2, y1:y2]
        lam = 1 - ((x2 - x1) * (y2 - y1) / (inputs.size(-1) * inputs.size(-2)))
    else:                                    # Mixup
        lam = np.random.beta(mixup_alpha, mixup_alpha)
        inputs = lam * inputs + (1 - lam) * inputs[idx]

    return inputs, target_a, target_b, lam


# ==========================================================
# [C] EMA
# ==========================================================
class ModelEMA:
    def __init__(self, model, decay=0.9998):
        self.ema = copy.deepcopy(model).eval()
        self.decay = decay
        self.updates = 0
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        with torch.no_grad():
            for ema_p, p in zip(self.ema.parameters(), model.parameters()):
                ema_p.data.mul_(d).add_(p.data, alpha=1 - d)
            for ema_b, b in zip(self.ema.buffers(), model.buffers()):
                ema_b.data.copy_(b.data)


# ==========================================================
# [D] Layer-wise LR Decay 파라미터 그룹 (timm ConvNeXt 네이밍 기준)
# ==========================================================
def convnext_layer_id(name, num_layers=13):
    """
    ConvNeXt-Tiny(depths=[3,3,9,3]) 전용 레이어 인덱싱.
      0        : stem
      1, 2     : stage0, stage1 (각각 한 덩어리)
      3 ~ 11   : stage2의 9개 블록 (블록당 하나씩)
      12       : stage3
      13       : head / norm
    공식 ConvNeXt LLRD 코드는 stage2가 27블록인 Base 기준(3블록씩 묶음)이라
    Tiny에 그대로 쓰면 인덱스가 5→12로 튀어 LR이 불연속이 된다. 그래서 직접 매핑.
    """
    if name.startswith('stem'):
        return 0
    if name.startswith('stages'):
        parts = name.split('.')
        stage_id = int(parts[1])
        is_down = (parts[2] == 'downsample')
        block_id = 0 if is_down else (int(parts[3]) if len(parts) > 3 and parts[3].isdigit() else 0)
        if stage_id in (0, 1):
            return stage_id + 1
        if stage_id == 2:
            return 3 + block_id          # downsample은 stage2 진입부 → 3
        return num_layers - 1            # stage 3 → 12
    return num_layers                    # head, norm_pre 등 → 13


def build_param_groups(model, base_lr, layer_decay=0.75, weight_decay=0.05, num_layers=13):
    groups = {}
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        no_decay = (param.ndim == 1 or name.endswith('.bias'))
        lid = convnext_layer_id(name, num_layers)
        key = (lid, no_decay)
        if key not in groups:
            groups[key] = {
                'params': [],
                'lr': base_lr * (layer_decay ** (num_layers - lid)),
                'weight_decay': 0.0 if no_decay else weight_decay,
            }
        groups[key]['params'].append(param)

    out = list(groups.values())
    lrs = sorted({g['lr'] for g in out})
    print(f"LLRD 적용: 파라미터 그룹 {len(out)}개 | LR 범위 {lrs[0]:.2e} ~ {lrs[-1]:.2e}")
    return out


# ==========================================================
# [E] 평가
# ==========================================================
@torch.no_grad()
def evaluate(model, loader, criterion, device, tta=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        with autocast('cuda'):
            outputs = model(inputs)
            if tta:
                outputs = (outputs + model(torch.flip(inputs, dims=[3]))) / 2.0
            loss = criterion(outputs, labels)
        total_loss += loss.item() * inputs.size(0)
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
    return total_loss / total, correct / total


# ==========================================================
# [F] 학습
# ==========================================================
def train_model(model, train_loader, clean_loader, val_loader,
                criterion, optimizer, scheduler, scaler, device, epochs, save_path):
    best_acc, best_epoch = 0.0, 0
    ema = ModelEMA(model, decay=0.9998)
    history = {'clean_train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

    print("\n[v2] 규제 강화 파인튜닝 시작")
    print("TrivialAugment + Mixup/CutMix + DropPath0.3 + LLRD0.75\n")

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        running = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/{epochs:03d} [Train]")

        for inputs, labels in pbar:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            inputs, ta, tb, lam = mix_batch(inputs, labels)

            with autocast('cuda'):
                outputs = model(inputs)
                loss = lam * criterion(outputs, ta) + (1 - lam) * criterion(outputs, tb)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            ema.update(model)

            running += loss.item() * inputs.size(0)
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        train_loss = running / len(train_loader.dataset)
        _, clean_acc = evaluate(ema.ema, clean_loader, criterion, device, tta=False)
        val_loss, val_acc = evaluate(ema.ema, val_loader, criterion, device, tta=True)
        scheduler.step()

        history['clean_train_acc'].append(clean_acc)
        history['val_acc'].append(val_acc)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        gap = clean_acc - val_acc
        lr_now = max(g['lr'] for g in optimizer.param_groups)
        print(f"Epoch {epoch+1:03d} | {time.time()-t0:.0f}s | headLR {lr_now:.2e} | "
              f"CleanTrain {clean_acc:.4f} | Val {val_acc:.4f} | Gap {gap:+.4f}")

        if val_acc > best_acc:
            best_acc, best_epoch = val_acc, epoch + 1
            torch.save(ema.ema.state_dict(), save_path)
            print(f"  >> best 갱신 · 저장: {save_path}\n")
        else:
            print()

    print(f"\n종료. 최고 Val {best_acc:.4f} (Epoch {best_epoch})")
    return history


def plot_history(history, save_dir):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history['clean_train_acc'], label='Clean Train Acc')
    plt.plot(history['val_acc'], label='Val Acc (EMA+TTA)')
    plt.title('Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Loss')
    plt.legend()
    p = os.path.join(save_dir, "finetune_v2_log.png")
    plt.savefig(p)
    print(f"\n그래프 저장: {p}")
    try:
        plt.show()
    except Exception:
        pass


# ==========================================================
# 메인
# ==========================================================
if __name__ == "__main__":
    if MODE in ("leakcheck", "both"):
        run_leak_check(DATA_DIR, LEAK_REPORT_PATH)
        if MODE == "leakcheck":
            raise SystemExit(0)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"학습 장치: {device}")

    # ---- 증강 강화 ----
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.65, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.TrivialAugmentWide(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.25),
    ])
    val_transform = transforms.Compose([
        transforms.Resize(236),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_root = os.path.join(DATA_DIR, 'train')
    val_root = os.path.join(DATA_DIR, 'val')

    print("데이터셋 로딩 중...")
    train_dataset = datasets.ImageFolder(train_root, transform=train_transform)
    val_dataset = datasets.ImageFolder(val_root, transform=val_transform)

    clean_full = datasets.ImageFolder(train_root, transform=val_transform)
    random.seed(42)
    clean_idx = random.sample(range(len(clean_full)), min(3000, len(clean_full)))
    clean_dataset = Subset(clean_full, clean_idx)

    mk = lambda ds, sh: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=sh,
                                   num_workers=NUM_WORKERS,
                                   persistent_workers=(NUM_WORKERS > 0), pin_memory=True)
    train_loader = mk(train_dataset, True)
    clean_loader = mk(clean_dataset, False)
    val_loader = mk(val_dataset, False)

    NUM_CLASSES = len(train_dataset.classes)
    print(f"클래스 수: {NUM_CLASSES}")

    model = timm.create_model(
        'convnext_tiny.fb_in22k_ft_in1k',
        pretrained=True,
        num_classes=NUM_CLASSES,
        drop_path_rate=DROP_PATH,
    ).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    param_groups = build_param_groups(model, BASE_LR, LAYER_DECAY, WEIGHT_DECAY)
    optimizer = optim.AdamW(param_groups)

    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS)
    cosine = CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])
    scaler = GradScaler('cuda')

    history = train_model(model, train_loader, clean_loader, val_loader,
                          criterion, optimizer, scheduler, scaler,
                          device, EPOCHS, MODEL_SAVE_PATH)
    plot_history(history, DATA_DIR)